## MIMIC Dataset


Verify current working directory

In [17]:
import os
import sys
import pickle
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import cv2

assert (
    Path(os.getcwd()).name == "AutoPrognosis-Multimodal"
), f"It seems like your are in the wrong directory ({os.getcwd()}), make sure to change to the autoprognosis_m directory."

# sys.path.append(os.path.abspath(os.path.join("../")))
# sys.path.append(os.path.abspath(os.path.join("../..")))

In [18]:
# Define disease etc.
disease = "(6) Heart failure" # '(1) Hypertensive diseases', '(2) Ischaemic heart diseases', '(3) Chronic ischaemic heart disease', '(4) Cardiomyopathies diseases', '(5) Dysrhythmias diseases', '(6) Heart failure'
path_data = f"/data/wolf6245/src/mm_study/data/f_modelling/03_model_input/data-2024-12-19-01-23-23/{disease}"
image_path_source = "/data/wolf6245/src/mm_study/data/a_raw/MIMIC/MIMIC-CXR-JPG/files"
path_target = "/data/wolf6245/src/AutoPrognosis-Multimodal/data_mimic"

## Preprocess the data


In [19]:
# Load all the data
image_metadata = pd.read_csv("/data/wolf6245/src/mm_study/data/a_raw/MIMIC/MIMIC-CXR-JPG/cxr_jpg/metadata.csv.gz")
mimic_master_clean = pd.read_parquet("/data/wolf6245/src/mm_study/data/e_prepared/icd/mimic_master_clean.parquet")
y_fusion = pd.read_parquet(path_data + "/y_fusion_label_not_gt.parquet")
y_ecgs = pd.read_parquet(path_data + "/y_ecgs_only_label_not_gt.parquet")
y_images = pd.read_parquet(path_data + "/y_images_only_label_not_gt.parquet")
X_images = pd.read_parquet(path_data + "/X_images_only_label_not_gt.parquet")
y_tabular = pd.read_parquet(path_data + "/y_tabular_only_label_not_gt.parquet")
X_tabular = pd.read_parquet(path_data + "/X_tabular_only_label_not_gt.parquet")
X_fusion_images = pd.read_parquet(path_data + "/X_images_fusion_label_not_gt.parquet")
X_fusion_tabular = pd.read_parquet(path_data + "/X_tabular_fusion_label_not_gt.parquet")
X_ecgs = pd.read_parquet(path_data + "/X_ecgs_only_label_not_gt.parquet")
with open(path_data + "/train_test_vali_folds_fusion_label.pkl", "rb") as f:
    test_valid_fusion = pickle.load(f)
with open(path_data + "/train_test_vali_folds_tabular_label.pkl", "rb") as f:
    test_valid_tabular = pickle.load(f)
with open(path_data + "/train_test_vali_folds_images_label.pkl", "rb") as f:
    test_valid_images = pickle.load(f)
with open(path_data + "/train_test_vali_folds_ecgs_label.pkl", "rb") as f:
    test_valid_ecgs = pickle.load(f)

# Pre-filter metadata
old_size = image_metadata.shape[0]
image_metadata = image_metadata[image_metadata["ViewPosition"].isin(["AP", "PA"])]
image_metadata["subject_start"] = image_metadata["subject_id"].astype(str).str[:2]
image_metadata["image_path"] = image_metadata.apply(
    lambda x: image_path_source + "/" + f"p{x["subject_start"]}" + "/" + f"p{x["subject_id"]}" + "/" + f"s{x['study_id']}" + "/" + x["dicom_id"] + ".jpg",
    axis=1,
)
image_metadata["img_id"] = image_metadata["image_path"].apply(lambda x: path_target.split("/")[-1] + "/images/" + os.path.basename(x))
print(f"Filtered metadata shape: {image_metadata.shape}, from {old_size} to {image_metadata.shape[0]}")

# Get matches
study_id_hadm_id_match = mimic_master_clean[~mimic_master_clean.cxr_study_id.isna()][['subject_id', 'hadm_id', 'cxr_study_id']].drop_duplicates()

print(f"{image_metadata.subject_id.nunique()} subject_ids in metadata")
print(f"{mimic_master_clean.subject_id.nunique()} subject_ids in mimic_master_clean")
print(f"{y_fusion.subject_id.nunique()} subject_ids in y_fusion")
print(f"{y_images.subject_id.nunique()} subject_ids in y_images")
print(f"{y_tabular.subject_id.nunique()} subject_ids in y_tabular")

Filtered metadata shape: (243334, 15), from 377110 to 243334
63945 subject_ids in metadata
28262 subject_ids in mimic_master_clean
5280 subject_ids in y_fusion
26473 subject_ids in y_images
28252 subject_ids in y_tabular


In [20]:
# Get all subject ids to use
subject_ids_fusion = y_fusion.subject_id.unique()
print(f"Number of patients with fusion data: {len(subject_ids_fusion)}")

y_images_aux = y_images.copy()
y_images_aux = y_images_aux[y_images_aux.study_id.isin(image_metadata.study_id.unique())]
subject_ids_tabular_images = [s for s in y_tabular.subject_id.unique() if s in y_images_aux.subject_id.unique()]
print(f"Number of patients with tabular and image data: {len(subject_ids_tabular_images)}")

subject_ids_tabular_images_ecgs = [s for s in subject_ids_tabular_images if s in y_ecgs.subject_id.unique()]
print(f"Number of patients with tabular and ecg data: {len(subject_ids_tabular_images_ecgs)}")

Number of patients with fusion data: 5280
Number of patients with tabular and image data: 8760
Number of patients with tabular and ecg data: 7574


In [21]:
#TODO: For now we will use only the fusion data, later filter better to use subject_ids_tabular_images
study_id_hadm_id_match_to_use = study_id_hadm_id_match[study_id_hadm_id_match.subject_id.isin(y_fusion.subject_id.unique())]
metadata_df = image_metadata[image_metadata.study_id.isin(study_id_hadm_id_match_to_use.cxr_study_id.astype(int).unique())]
print(f"Filtered metadata shape: {metadata_df.shape}, from {image_metadata.shape[0]} to {metadata_df.shape[0]}, with {metadata_df.subject_id.nunique()} unique subject_ids from {image_metadata.subject_id.nunique()} unique subject_ids")
metadata_df = metadata_df.drop_duplicates(subset="subject_id")
print(f"Dropped duplicates, now {metadata_df.shape[0]} rows and {metadata_df.subject_id.nunique()} unique subject_ids")

X_tabular_to_use = X_fusion_tabular[X_fusion_tabular.subject_id.isin(metadata_df.subject_id.unique())].copy()
X_images_to_use = X_fusion_images[X_fusion_images.subject_id.isin(metadata_df.subject_id.unique())].copy()
y_to_use = y_fusion[y_fusion.subject_id.isin(metadata_df.subject_id.unique())].copy()
fold_to_use = test_valid_fusion

print(f"Using y_to_use: {y_to_use.shape}")
print(f"Using X_images_to_use: {X_images_to_use.shape}")
print(f"Using X_tabular_to_use: {X_tabular_to_use.shape}")

feature_columns = [c for c in X_tabular_to_use.columns if c not in ["subject_id", "img_id", "diagnostic", "study_id", "hadm_id"]]
print(f"Features: {len(feature_columns)}")

Filtered metadata shape: (5806, 15), from 243334 to 5806, with 5216 unique subject_ids from 63945 unique subject_ids
Dropped duplicates, now 5216 rows and 5216 unique subject_ids
Using y_to_use: (5216, 12)
Using X_images_to_use: (5216, 2)
Using X_tabular_to_use: (5216, 116)
Features: 115


In [22]:
# Standardise X_tabular_to_use
if False:
    from sklearn.preprocessing import StandardScaler
    X_features = X_tabular_to_use[feature_columns]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_features)
    X_scaled_df = pd.DataFrame(X_scaled, columns=feature_columns, index=X_tabular_to_use.index)
    X_standardised = X_tabular_to_use.copy()
    X_standardised[feature_columns] = X_scaled_df
else:
    X_standardised = X_tabular_to_use

# Replace missing values with column mean
X_standardised = X_standardised.fillna(X_standardised.mean())

In [23]:
if False:
    # Copy images to target directory
    image_paths = metadata_df.image_path.unique()
    image_path_target = path_target + "/images"
    os.makedirs(image_path_target, exist_ok=True)
    print(f"Copying {len(image_paths)} images to {image_path_target}...")
    for image_path in tqdm(image_paths):
        target_path = image_path_target + "/" + os.path.basename(image_path)
        os.system(f"cp {image_path} {target_path}")

    # Resize images
    def resize_images(data_dir, target_dims, file_type="png"):
        data_dir = Path(data_dir)
        image_dir = data_dir / "images"
        for image_path in tqdm(list(image_dir.rglob(f"*.{file_type}"))):
            img = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
            img_resized = cv2.resize(img, target_dims)
            cv2.imwrite(str(image_path), img_resized)
    target_dims = (448, 448)
    print(f"Resizing images to {target_dims}...")
    resize_images(path_target, target_dims, "jpg")
    print("Images resized.")

In [24]:
# Create splits
if True:
    y_fusion_aux = y_to_use[["subject_id", disease]].copy()
    y_fusion_aux.rename(columns={disease: "diagnostic"}, inplace=True)
    y_fusion_aux["diagnostic"] = y_fusion_aux["diagnostic"].astype(int).astype(str)
    split_mapping = {
        "train": 0,
        "val": 2,
        "test": 1,
    }
    for fold_id, fold in enumerate(fold_to_use):
        print(f"Fold {fold_id}")
        storage_path = f"{path_target}/{disease}/folds/fold_{fold_id}"
        os.makedirs(storage_path, exist_ok=True)
        for split_name, split_id in split_mapping.items():
            print(f"-- Split {split_name}")
            split_subject_ids = fold[split_id]
            df_aux = X_standardised[X_standardised.subject_id.isin(split_subject_ids)].copy()
            df_aux_old_shape = df_aux.shape[0]
            df_aux = df_aux.merge(
                y_fusion_aux,
                on="subject_id",
                how="left",
            )
            assert df_aux.shape[0] == df_aux_old_shape, f"Shape mismatch: {df_aux.shape[0]} != {df_aux_old_shape}"
            df_aux_old_shape = df_aux.shape[0]
            df_aux = df_aux.merge(
                metadata_df[["subject_id", "img_id"]],
                on="subject_id",
                how="left",
            )
            assert df_aux.shape[0] == df_aux_old_shape, f"Shape mismatch: {df_aux.shape[0]} != {df_aux_old_shape}"
            
            # print(f"Shape of df_aux: {df_aux.shape}, subject_ids: {df_aux.subject_id.nunique()}, img_ids: {df_aux.img_id.nunique()}")
            # print(f"Ratio positive: {round(df_aux[df_aux.diagnostic == 1].shape[0] / df_aux.shape[0], 2)}")

            df_aux.to_csv(
                os.path.join(storage_path, f"{split_name}.csv"),
                index=False,
            )

Fold 0
-- Split train


-- Split val
-- Split test
Fold 1
-- Split train
-- Split val
-- Split test
Fold 2
-- Split train
-- Split val
-- Split test
Fold 3
-- Split train
-- Split val
-- Split test
Fold 4
-- Split train
-- Split val
-- Split test
